In [1]:
using CSV, DataFrames, Statistics,Printf, StatsBase,  Plots

include("functions.jl")
data_dir = "output"  
burnin   = 500_000
thin     = 10
chain_ids = 1:3
outfile = joinpath(data_dir, "rhat_summary.csv")
re = r"^samples_sim_(\d+)_chain_([123])\.csv$"
sim_ids = readdir(data_dir) .|> x -> match(re, x)
sim_ids = sort!(unique(parse(Int, m.captures[1]) for m in sim_ids if m !== nothing))

function load_chain_df(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

all_sims_gr = Vector{Dict{String,Float64}}()

for sim in sim_ids
    dfs = DataFrame[]
    for c in chain_ids
        fpath = joinpath(data_dir, "samples_sim_$(sim)_chain_$(c).csv")
        push!(dfs, load_chain_df(fpath; burnin=burnin, thin=thin))
    end

    params = String.(names(dfs[1]))
    n_keep = nrow(dfs[1])

    gr_dict = Dict{String,Float64}()
    for p in params

        mat = Array{Float64}(undef, length(chain_ids), n_keep)
        for (i, df) in enumerate(dfs)
            mat[i, :] = Float64.(df[!, p])
        end
        gr_dict[p] = rhat_gelman_rubin(mat)
    end

    push!(all_sims_gr, gr_dict)
end

write_gelman_rubin_csv(outfile, all_sims_gr)


In [2]:
using DelimitedFiles
using Statistics
using StatsBase
using Random
using Printf


const OUTPUT_DIR        = "output"
const N_CHAINS          = 3
const BURN_IN_SAMPLES   = 500_000    
const THIN_SAMPLES      = 10    

const BURN_IN_LOGLIK    = 0
const THIN_LOGLIK       = 1

# memoryless: ["beta","alpha","gamma"]
# sliding:    ["beta","alpha","gamma","k_max"]
# powerlaw:   ["beta","alpha","gamma","lambda_P"]
# exponential:["beta","alpha","gamma","lambda_E"]
# reciprocal: ["beta","alpha","gamma","lambda_R"]
param_header = ["beta","alpha","gamma"]

const N_DRAWS = 1000

function read_csv_matrix(path::String)
    data, hdr = DelimitedFiles.readdlm(path, ',', header=true)
    M = Matrix{Float64}(data)
    header = hdr === nothing ? nothing : vec(collect(hdr))
    return M, header
end

function detect_sim_ids(outdir::String)
    files = filter(isfile, readdir(outdir; join=true))
    sims = Int[]
    for f in files
        m = match(r"samples_sim_(\d+)_chain_(\d+)\.csv", f)
        if m !== nothing
            push!(sims, parse(Int, m.captures[1]))
        end
    end
    return unique(sort(sims))
end

function load_samples_for_sim(sim_id::Int, n_chains::Int, outdir::String)
    mats = Matrix{Float64}[]
    for c in 1:n_chains
        path = joinpath(outdir, "samples_sim_$(sim_id)_chain_$(c).csv")
        if !isfile(path)
            @warn "Missing samples file: $path"; continue
        end
        M, _ = read_csv_matrix(path)
        if size(M,1) <= BURN_IN_SAMPLES
            @warn "File $path has only $(size(M,1)) rows (<= burn-in)."
            continue
        end
        idxs = collect(BURN_IN_SAMPLES+1:THIN_SAMPLES:size(M,1))
        push!(mats, M[idxs, :])
    end
    return isempty(mats) ? Array{Float64}(undef, 0, 0) : vcat(mats...)
end

function load_loglik_list_for_sim(sim_id::Int, n_chains::Int, outdir::String)
    ll_list = Matrix{Float64}[]
    for c in 1:n_chains
        path = joinpath(outdir, "loglik_sim_$(sim_id)_chain_$(c).csv")
        if !isfile(path)
            @warn "Missing loglik file: $path"; continue
        end
        M, _ = read_csv_matrix(path)
        if BURN_IN_LOGLIK > 0 || THIN_LOGLIK > 1
            idxs = collect(BURN_IN_LOGLIK+1:THIN_LOGLIK:size(M,1))
            M = M[idxs, :]
        end
        push!(ll_list, M)
    end
    return ll_list
end

function logmeanexp_columnwise(L::AbstractMatrix{<:Real})
    S, T = size(L)
    out = Vector{Float64}(undef, T)
    for j in 1:T
        col = L[:, j]
        m = maximum(col)
        out[j] = m + log(sum(exp.(col .- m)) / S)
    end
    return out
end

function compute_waic_from_list(all_loglik_aug_vecs::Vector{Matrix{Float64}})
    combined_logliks = vcat(all_loglik_aug_vecs...)   # S×T
    if any(isinf, combined_logliks)
        @warn "Encountered infinite log-likelihoods. WAIC will be -Inf."
        return -Inf
    end
    lppd = sum(logmeanexp_columnwise(combined_logliks))
    pwaic = sum(var(combined_logliks, dims=1))
    return -2 * lppd + 2 * pwaic
end

function summarize_posterior(samples::AbstractMatrix{<:Real}, header_cont::Vector{String})
    summary_dict = Dict{String, Any}()
    for (i, p_name) in enumerate(header_cont)
        param_samples = samples[:, i]
        if p_name == "k_max"
            k = mode(round.(Int, param_samples))
            freq = count(==(k), round.(Int, param_samples)) / length(param_samples)
            summary_dict[p_name] = Dict("mode" => k, "frequency" => freq)
        else
            med = median(param_samples)
            q = quantile(param_samples, [0.025, 0.975])
            summary_dict[p_name] = Dict("median" => med, "95%CI_low" => q[1], "95%CI_high" => q[2])
        end
    end
    return summary_dict
end

function write_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        DelimitedFiles.writedlm(io, M, ',')
    end
end

function write_summary_csv(path::String, summary_data::Vector{Dict{String,Any}})
    open(path, "w") do io
        println(io, "Dataset,Parameter,Median_or_Mode,CI_Low_or_Frequency,CI_High")
        for (dataset_idx, dict) in enumerate(summary_data)
            for (param, vals) in dict
                if haskey(vals, "mode")
                    println(io, "$dataset_idx,$param,$(vals["mode"]),$(vals["frequency"]),")
                else
                    println(io, "$dataset_idx,$param,$(vals["median"]),$(vals["95%CI_low"]),$(vals["95%CI_high"])")
                end
            end
        end
    end
end

function write_waic_csv(path::String, waic_vec::Vector{Float64})
    open(path, "w") do io
        println(io, "Dataset,WAIC")
        for (i, v) in enumerate(waic_vec)
            println(io, "$i,$v")
        end
    end
end

Random.seed!(2025)

sim_ids = detect_sim_ids(OUTPUT_DIR)
isempty(sim_ids) && error("No samples_sim_*_chain_*.csv found in $(OUTPUT_DIR)")

waic_vec = Float64[]
posterior_summaries = Dict{String,Any}[]

for sim in sim_ids
    @info "Processing dataset $sim ..."
    samples_bt = load_samples_for_sim(sim, N_CHAINS, OUTPUT_DIR)
    if size(samples_bt,1) == 0
        @warn "No samples for dataset $sim; skipping"
        push!(waic_vec, NaN)
        push!(posterior_summaries, Dict{String,Any}())
        continue
    end

    push!(posterior_summaries, summarize_posterior(samples_bt, param_header))

    ll_list = load_loglik_list_for_sim(sim, N_CHAINS, OUTPUT_DIR)
    push!(waic_vec, isempty(ll_list) ? NaN : compute_waic_from_list(ll_list))

    n_rows = size(samples_bt, 1)
    replace_flag = n_rows < N_DRAWS
    draw_indices = sample(1:n_rows, N_DRAWS; replace=replace_flag)
    draw_params  = samples_bt[draw_indices, :]

    draws_outfile = joinpath(OUTPUT_DIR, @sprintf("posterior_draws_sim_%d.csv", sim))
    hdr = vcat(["row_index"], param_header)
    Mout = hcat(Float64.(draw_indices), draw_params)
    write_csv(draws_outfile, hdr, Mout)
end

write_waic_csv(joinpath(OUTPUT_DIR, "waic_by_dataset.csv"), waic_vec)
write_summary_csv(joinpath(OUTPUT_DIR, "posterior_summary_by_dataset.csv"), posterior_summaries)

@info "Done. Outputs written in $(OUTPUT_DIR)"


[ Info: Processing dataset 1 ...
[ Info: Processing dataset 2 ...
[ Info: Processing dataset 3 ...
[ Info: Processing dataset 4 ...
[ Info: Processing dataset 5 ...
[ Info: Processing dataset 6 ...
[ Info: Processing dataset 7 ...
[ Info: Processing dataset 8 ...
[ Info: Processing dataset 9 ...
[ Info: Processing dataset 10 ...
[ Info: Processing dataset 11 ...
[ Info: Processing dataset 12 ...
[ Info: Processing dataset 13 ...
[ Info: Processing dataset 14 ...
[ Info: Processing dataset 15 ...
[ Info: Processing dataset 16 ...
[ Info: Processing dataset 17 ...
[ Info: Processing dataset 18 ...
[ Info: Processing dataset 19 ...
[ Info: Processing dataset 20 ...
[ Info: Done. Outputs written in output


In [3]:
using Random, Distributions, Statistics, Printf, DelimitedFiles
using Plots
Random.seed!(2025) 
# ---------- helpers ----------
function clamp01(x)
    x < 0 ? 0.0 : (x > 1 ? 1.0 : x)
end

# a_t = 1 - (1 - ψ_t)^(1/α)
function baseline_alarm(psi, alpha)
    psi = clamp01(psi)
    alpha <= 0 && error("alpha must be > 0")
    1 - (1 - psi)^(1/alpha)
end

# ψ_t definitions (Istar_hist is a Vector of counts)
psi_memoryless(Istar_hist, N) = isempty(Istar_hist) ? 0.0 : Istar_hist[end] / N

function psi_sliding(Istar_hist, N, k_max)
    L = length(Istar_hist)
    L == 0 && return 0.0
    m = min(k_max, L)
    mean(@view(Istar_hist[(L - m + 1):L])) / N
end

function psi_powerlaw(Istar_hist, N, lambda_P)
    L = length(Istar_hist)
    L == 0 && return 0.0
    # weights j = 1..L, most recent has j=1
    num  = 0.0
    @inbounds for j in 1:L
        w = j^(-lambda_P)
        num  += w * Istar_hist[end - j + 1]
    end
    num / N   # unnormalized (consistent with your compute_psi!)
end

function psi_exponential(Istar_hist, N, lambda_E)
    L = length(Istar_hist)
    L == 0 && return 0.0
    # weights j = 0..L-1, most recent j=0
    num  = 0.0
    @inbounds for j in 0:(L - 1)
        w = exp(-lambda_E * j)
        num  += w * Istar_hist[end - j]
    end
    num / N   # unnormalized (consistent with your compute_psi!)
end

function psi_reciprocal(Istar_hist, N, lambda_R)
    lambda_R <= 0 && error("lambda_R must be > 0")
    L = length(Istar_hist)
    L == 0 && return 0.0
    num = 0.0
    @inbounds for j in 0:(L - 1)
        w = 1.0 / (1.0 + lambda_R * j)   # ensures w2 = 0.5 when lambda_R = 0.5
        num += w * Istar_hist[end - j]
    end
    num / N
end

# ---------- simulate one epidemic ----------
function simulate_epidemic(N, I0, tau, beta, alpha, rateI;
                           mechanism::Symbol = :memoryless,
                           k_max::Union{Nothing,Int}=nothing,
                           lambda_P::Union{Nothing,Float64}=nothing,
                           lambda_E::Union{Nothing,Float64}=nothing,
                           lambda_R::Union{Nothing,Float64}=nothing)

    S      = zeros(Int, tau + 1)
    I      = zeros(Int, tau + 1)
    Istar  = zeros(Int, tau)
    Rstar  = zeros(Int, tau)
    psi    = zeros(Float64, tau)
    alarm  = zeros(Float64, tau)
    probSI = zeros(Float64, tau)
    probIR = 1 - exp(-rateI)

    S[1] = N - I0
    I[1] = I0

    # t = 1
    psi[1]   = 0.0
    alarm[1] = baseline_alarm(psi[1], alpha)
    pSI      = clamp01(1 - exp(-beta * (1 - alarm[1]) * (I[1] / N)))
    probSI[1] = pSI
    Istar[1] = rand(Binomial(S[1], pSI))
    Rstar[1] = rand(Binomial(I[1], probIR))
    S[2]     = S[1] - Istar[1]
    I[2]     = I[1] + Istar[1] - Rstar[1]

    # t = 2..tau
    for t in 2:tau
        hist = @view Istar[1:(t-1)]
        ψt = if mechanism === :memoryless
            psi_memoryless(hist, N)
        elseif mechanism === :sliding
            isnothing(k_max) && error("k_max must be provided for :sliding")
            psi_sliding(hist, N, k_max)
        elseif mechanism === :powerlaw
            isnothing(lambda_P) && error("lambda_P must be provided for :powerlaw")
            psi_powerlaw(hist, N, lambda_P)
        elseif mechanism === :exponential
            isnothing(lambda_E) && error("lambda_E must be provided for :exponential")
            psi_exponential(hist, N, lambda_E)
        elseif mechanism === :reciprocal
            isnothing(lambda_R) && error("lambda_R must be provided for :reciprocal")
            psi_reciprocal(hist, N, lambda_R)
        else
            error("Unknown mechanism: $mechanism")
        end
        psi[t]   = ψt
        alarm[t] = baseline_alarm(ψt, alpha)
        pSI      = clamp01(1 - exp(-beta * (1 - alarm[t]) * (I[t] / N)))
        probSI[t] = pSI

        Istar[t] = rand(Binomial(S[t], pSI))
        Rstar[t] = rand(Binomial(I[t], probIR))

        S[t+1] = S[t] - Istar[t]
        I[t+1] = I[t] + Istar[t] - Rstar[t]
    end

    return Dict(
        :Istar => Istar,
        :Rstar => Rstar,
        :S     => S,
        :I     => I,
        :psi   => psi,
        :alarm => alarm,
        :probSI => probSI,
        :probIR => probIR
    )
end

# ---------- write CSV with header ----------
function write_matrix_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        for i in 1:size(M,1)
            println(io, join(M[i, :], ","))
        end
    end
end

OUTPUT_DIR = "output"
TAU = 50
N_pop, I0 = 1_000_000, 10
MECHANISM = :powerlaw
params = ["beta","alpha","gamma","k_max","lambda_P","lambda_E","lambda_R"]

function read_draws(path)
    data,hdr = readdlm(path,',',header=true)
    Matrix{Float64}(data),vec(hdr)
end

function idxmap(header,names)
    H=Dict(n=>i for (i,n) in enumerate(header))
    Dict(n=>get(H,n,nothing) for n in names)
end

function summarize(M)
    τ=size(M,2); med=Float64[]; lo=Float64[]; hi=Float64[]
    for t=1:τ
        col=M[:,t]; push!(med,median(col))
        q=quantile(col,[0.025,0.975]); push!(lo,q[1]); push!(hi,q[2])
    end
    med,lo,hi
end

function write_csv(path,hdr,rows)
    open(path,"w") do io
        println(io,join(hdr,",")); foreach(r->println(io,join(r,",")),rows)
    end
end

function avg_series(list)
    τ=length(list[1]); [mean([s[t] for s in list]) for t=1:τ]
end

sim_ids=sort(parse.(Int,replace.(filter(f->occursin(r"posterior_draws_sim_\d+\.csv",f),readdir(OUTPUT_DIR)),
                                     r"[^\d]"=>"")))
istarM=[]; istarL=[]; istarH=[]; alarmM=[]; alarmL=[]; alarmH=[]

for sim in sim_ids
     @info "Processing dataset $sim ..."
    M,hdr=read_draws("$(OUTPUT_DIR)/posterior_draws_sim_$(sim).csv")
    col=idxmap(hdr,params); n=size(M,1)
    istar=zeros(n,TAU); alarm=zeros(n,TAU)
    for k=1:n
        β,α,γ=M[k,col["beta"]],M[k,col["alpha"]],M[k,col["gamma"]]
        kmax=col["k_max"]==nothing ? nothing : round(Int,M[k,col["k_max"]])
        λP=col["lambda_P"]==nothing ? nothing : M[k,col["lambda_P"]]
        λE=col["lambda_E"]==nothing ? nothing : M[k,col["lambda_E"]]
        λR=col["lambda_R"]==nothing ? nothing : M[k,col["lambda_R"]]
        simres=simulate_epidemic(N_pop,I0,TAU,β,α,γ;mechanism=:memoryless,
                                 k_max=kmax,lambda_P=λP,lambda_E=λE,lambda_R=λR)
        istar[k,:]=simres[:Istar]; alarm[k,:]=simres[:alarm]
    end
    m,l,h=summarize(istar); write_csv("$(OUTPUT_DIR)/Istar_stats_sim_$(sim).csv",
        ["Day","median","ci_low","ci_high"],[[t,m[t],l[t],h[t]] for t=1:TAU])
    m,l,h=summarize(alarm); write_csv("$(OUTPUT_DIR)/alarm_stats_sim_$(sim).csv",
        ["Day","median","ci_low","ci_high"],[[t,m[t],l[t],h[t]] for t=1:TAU])
    push!(istarM,m);push!(istarL,l);push!(istarH,h)
    push!(alarmM,m);push!(alarmL,l);push!(alarmH,h)
end

DATA_DIR   = "data"

function average_files(pattern)
    files = sort(filter(f -> occursin(pattern, f), readdir(OUTPUT_DIR; join=true)))
    matrices = [DelimitedFiles.readdlm(f, ',', header=true)[1] for f in files]
    reduce(+, matrices) ./ length(matrices)
end

function add_truth_and_save(meanmat, truthfile, truthcolname, outfile)
    data, hdr = readdlm(joinpath(DATA_DIR, truthfile), ',', header=true)
    H = Dict(n=>i for (i,n) in enumerate(vec(hdr)))
    truth = data[:, H[truthcolname]]
    out = hcat(meanmat, truth)
    open(joinpath(OUTPUT_DIR, outfile), "w") do io
        println(io, "Day,median,ci_low,ci_high,truth")
        writedlm(io, out, ',')
    end
end

# Istar
istar_mean = average_files("Istar_stats_sim_")
add_truth_and_save(istar_mean, "memoryless_mean_incidence.csv", "mean_Istar", "Istar_stats_mean.csv")

# Alarm
alarm_mean = average_files("alarm_stats_sim_")
add_truth_and_save(alarm_mean, "memoryless_mean_alarm.csv", "mean_alarm", "alarm_stats_mean.csv")

[ Info: Processing dataset 1 ...
[ Info: Processing dataset 2 ...
[ Info: Processing dataset 3 ...
[ Info: Processing dataset 4 ...
[ Info: Processing dataset 5 ...
[ Info: Processing dataset 6 ...
[ Info: Processing dataset 7 ...
[ Info: Processing dataset 8 ...
[ Info: Processing dataset 9 ...
[ Info: Processing dataset 10 ...
[ Info: Processing dataset 11 ...
[ Info: Processing dataset 12 ...
[ Info: Processing dataset 13 ...
[ Info: Processing dataset 14 ...
[ Info: Processing dataset 15 ...
[ Info: Processing dataset 16 ...
[ Info: Processing dataset 17 ...
[ Info: Processing dataset 18 ...
[ Info: Processing dataset 19 ...
[ Info: Processing dataset 20 ...


In [4]:
using CSV, DataFrames, Statistics, StatsBase
using Plots

# ========= 配置 =========
data_dir  = "output"         # 你的样本目录：包含 samples_sim_{k}_chain_{1..3}.csv
burnin    = 500_000
thin      = 10
chain_ids = 1:3
save_root = joinpath(data_dir, "viz")
isdir(save_root) || mkpath(save_root)

default(fmt = :png, size=(1100,600))  # 全局图尺寸，按需调整

# 自动发现可用的 sim（根据文件名）
re = r"^samples_sim_(\d+)_chain_([123])\.csv$"
sim_matches = readdir(data_dir) .|> x -> match(re, x)
sim_ids = sort!(unique(parse(Int, m.captures[1]) for m in sim_matches if m !== nothing))
isempty(sim_ids) && error("在 $(data_dir) 下未发现 samples_sim_{k}_chain_{1..3}.csv")

# -------- 只画一个 sim：在这里指定 --------
sim = first(sim_ids)         # 改成你想画的编号，如：sim = 3
# ----------------------------------------

# ========= 工具函数 =========
function load_chain(path::String; burnin::Int, thin::Int)
    df = CSV.read(path, DataFrame)
    idx = (burnin + 1):thin:nrow(df)
    return df[idx, :]
end

# Trace plot：三条链叠加；图例放在 x 轴下方（图外）
function plot_trace(dfs::Vector{DataFrame}, param::String; max_points=500000)
    plt = plot(title="Trace: $param",
               xlabel="Iteration", ylabel=param,
               legend=:outerbottom, orientation=:horizontal, legendfont=font(10))
    for (i, df) in enumerate(dfs)
        vals = df[!, param]
        n = min(length(vals), max_points)
        plot!(1:n, vals[1:n], label="chain $i", alpha=0.8)
    end
    return plt
end

# 相关系数热图（合并三条链后计算 Pearson 相关；为提速默认每链下采样 every_k）
function correlation_heatmap(dfs::Vector{DataFrame}; every_k::Int=10)
    # 合并三条链并下采样
    df_subs = DataFrame[]
    for df in dfs
        idx = 1:every_k:nrow(df)
        push!(df_subs, df[idx, :])
    end
    big = vcat(df_subs...)
    # 只取数值列
    numcols = [nm for nm in names(big) if eltype(big[!, nm]) <: Real]
    isempty(numcols) && error("找不到数值型参数列用于相关性计算")
    big = big[:, numcols]
    # 计算相关矩阵（列相关）
    M = Matrix{Float64}(big)
    C = cor(M; dims=1)
    # 动态尺寸：参数多时放大画布
    sz = (max(600, 70*length(numcols)), max(600, 70*length(numcols)))
    plt = heatmap(C;
        c = :balance, clim = (-1, 1), colorbar_title = "corr",
        xticks = (1:length(numcols), string.(numcols)),
        yticks = (1:length(numcols), string.(numcols)),
        size = sz,
        title = "Parameter correlation (Pearson), subsample every $every_k"
    )
    return plt
end


function pairwise_scatter(dfs::Vector{DataFrame}, params::Vector{String};
                          every_k::Int=20)
    df_subs = DataFrame[]
    for df in dfs
        idx = 1:every_k:nrow(df)
        push!(df_subs, df[idx, :])
    end
    big = vcat(df_subs...)
    P = length(params)
    lay = (P, P)
    plt = plot(layout = lay, size = (140*P, 140*P), legend = false)
    for (i, pi) in enumerate(params)
        for (j, pj) in enumerate(params)
            ax = (i-1)*P + j
            if i == j
                histogram!(plt[ax], big[!, pi], bins = 30, color=:steelblue, alpha=0.7)
            else
                scatter!(plt[ax], big[!, pj], big[!, pi], markersize = 2, alpha = 0.5)
            end
            
            # 只移除x轴数字刻度
            plot!(plt[ax], xticks=false)
            
            # 设置轴标签
            i == P && xlabel!(plt[ax], pj)
            j == 1 && ylabel!(plt[ax], pi)
        end
    end
    return plt
end
# ========= 主流程（只画一个 sim） =========
# 读入该 sim 的三条链
dfs = DataFrame[]
for c in chain_ids
    f = joinpath(data_dir, "samples_sim_$(sim)_chain_$(c).csv")
    push!(dfs, load_chain(f; burnin=burnin, thin=thin))
end

# 参数名（假设三条链列名一致）
params = String.(names(dfs[1]))
outdir = joinpath(save_root, "sim_$(sim)")
isdir(outdir) || mkpath(outdir)

# (A) 每个参数：Trace plot
for p in params
    fig = plot_trace(dfs, p; max_points=500000)
    savefig(fig, joinpath(outdir, "trace_$(p).png"))
end

if length(params) <= 8
    pair = pairwise_scatter(dfs, params; every_k=20)
    savefig(pair, joinpath(outdir, "pair_scatter.png"))
end

println("sim $sim done → traces + correlation visualizations saved in $outdir")


sim 1 done → traces + correlation visualizations saved in output/viz/sim_1
